In [ ]:
# Embeddings: embed every corpus row (train + test, by row id) once per embedder after
# textprep.preprocess_text. Labels and the split are NOT stored; consumers join by row id with
# embeddings.load_artifact and verify fingerprints (notebooks/AGENTS.md).
import os
import time
from pathlib import Path

from dotenv import load_dotenv

import numpy as np
import pandas as pd

load_dotenv()
os.chdir(os.getenv("ROOT"))

from notebooks import embeddings, splits

In [ ]:
SPLIT_MANIFEST_CSV = Path(os.environ["SPLIT_MANIFEST_CSV"])
SPLIT_TRAIN_CSV = Path(os.environ["SPLIT_TRAIN_CSV"])
SPLIT_TEST_CSV = Path(os.environ["SPLIT_TEST_CSV"])
SPLIT_MANIFEST_SHA256 = os.environ["SPLIT_MANIFEST_SHA256"]  # pinned digest printed by Train-Test split
EMBEDDINGS_DIR = Path(os.environ["EMBEDDINGS_DIR"])
COTOV_CSV = os.getenv("COTOV_CSV")
FROZEN_SEED = 42
PARITY_ROWS = 64

# Smoke runs only: embed the first N rows (by id); the artifacts are marked NON-PRODUCTION.
SAMPLE_ROWS = int(os.getenv("EMBED_SAMPLE_ROWS", "0"))
NON_PRODUCTION = SAMPLE_ROWS > 0
selected = [n.strip() for n in os.getenv("EMBEDDERS", "").split(",") if n.strip()] or None
embedders = embeddings.load_embedders(names=selected)
DEVICE = embeddings.resolve_device(os.getenv("EMBED_DEVICE") or None)
print("embedders:", list(embedders), "| device (transformers):", DEVICE, "| NON-PRODUCTION:", NON_PRODUCTION)

In [ ]:
# Corpus rows: verified against the pinned manifest before anything is embedded.
train = pd.read_csv(SPLIT_TRAIN_CSV)
test = pd.read_csv(SPLIT_TEST_CSV)
manifest = splits.read_manifest(SPLIT_MANIFEST_CSV)
full = pd.concat([train, test], ignore_index=True)
splits.verify_manifest(full, manifest, expected_seed=FROZEN_SEED, expected_manifest_sha256=SPLIT_MANIFEST_SHA256)
cols = manifest.attrs["metadata"]["columns"]
full["_row_id"] = full[cols["id"]].astype(str)
assert full["_row_id"].is_unique
assert set(full["_row_id"]) == set(manifest["row_id"]), "corpus rows differ from the manifest"
corpus = full.sort_values("_row_id", kind="mergesort")  # canonical id order
corpus_ids = corpus["_row_id"].tolist()
corpus_texts = corpus[cols["text"]].tolist()
print(f"manifest verified ({SPLIT_MANIFEST_SHA256[:12]}...): {len(corpus_ids)} corpus rows")

In [ ]:
# Corpora to embed: name -> (row ids, raw texts).
# Cotov (external, English only): rows with lang == 'en', text column `full_text`. The file has no
# id column, so the row id is `cotov-<0-based row position in the CSV>` (an `id` column is used
# instead when present); stable as long as the CSV is unchanged (the raw-text digest binds it).
corpora = {"corpus": (corpus_ids, corpus_texts)}
if COTOV_CSV and os.path.exists(COTOV_CSV):
    cotov = pd.read_csv(COTOV_CSV)
    cotov["_row_id"] = cotov["id"].astype(str) if "id" in cotov.columns else [f"cotov-{i}" for i in range(len(cotov))]
    cotov = cotov[cotov["lang"] == "en"].sort_values("_row_id", kind="mergesort")
    assert cotov["_row_id"].is_unique
    corpora["cotov"] = (cotov["_row_id"].tolist(), cotov["full_text"].tolist())
else:
    print("BLOCKED: Cotov dataset unavailable (COTOV_CSV); only the corpus is embedded")
if NON_PRODUCTION:
    corpora = {k: (ids[:SAMPLE_ROWS], texts[:SAMPLE_ROWS]) for k, (ids, texts) in corpora.items()}
print({k: len(v[0]) for k, v in corpora.items()})

In [ ]:
# Embed every corpus with every embedder; an identical existing artifact is verified and reused.
report = []
for name, emb in embedders.items():
    fp = embeddings.fingerprint(emb)
    if fp.get("revision") == embeddings.UNRESOLVED:
        print(f"WARNING: {name}: revision unresolved; consumers fail closed unless they allow it")
    for corpus_name, (ids, texts) in corpora.items():
        stem = embeddings.artifact_name(corpus_name, name)
        t0 = time.time()
        expected_sha = embeddings.texts_sha256(ids, texts)
        try:
            art = embeddings.load_artifact(EMBEDDINGS_DIR, stem, expected_row_ids=ids, expected_fingerprint=fp,
                                           expected_texts_sha256=expected_sha, allow_non_production=NON_PRODUCTION)
            assert art.meta["non_production"] == NON_PRODUCTION and len(art) == len(ids)
            status = "reused"
        except embeddings.ArtifactCorruptError as e:
            if (EMBEDDINGS_DIR / f"{stem}.json").exists():
                raise  # an existing but broken artifact is never overwritten silently
            matrix = embeddings.embed(texts, emb, device=DEVICE)
            meta = embeddings.build_meta(emb, ids, texts, corpus=corpus_name, device=DEVICE, fp=fp,
                                         non_production=NON_PRODUCTION)
            embeddings.write_artifact(EMBEDDINGS_DIR, stem, ids, matrix, meta)
            art = embeddings.load_artifact(EMBEDDINGS_DIR, stem, expected_row_ids=ids, expected_fingerprint=fp,
                                           expected_texts_sha256=expected_sha, allow_non_production=NON_PRODUCTION)
            assert np.array_equal(art.select(ids), matrix)
            status = "written"
        seconds = time.time() - t0
        report.append({"artifact": stem, "status": status, "shape": art.shape, "seconds": round(seconds, 1),
                       "fingerprint": fp, "npy_sha256": art.meta["npy_sha256"]})
        print(f"{stem}: {status} {art.shape} in {seconds:.1f}s | fingerprint {fp}")

In [ ]:
# Parity: re-embed a fixed sample with another batch size; GloVe must agree exactly, transformers
# within rtol (embedders.toml [parity]). A stored artifact must also equal a fresh embedding.
embedded_ids, embedded_texts = corpora["corpus"]  # the rows actually embedded (a prefix in smoke runs)
sample = list(range(0, len(embedded_ids), max(1, len(embedded_ids) // PARITY_ROWS)))[:PARITY_ROWS]
sample_ids = [embedded_ids[i] for i in sample]
sample_texts = [embedded_texts[i] for i in sample]
for name, emb in embedders.items():
    result = embeddings.parity_check(emb, sample_texts, alt_batch_size=7, device=DEVICE)
    stored = embeddings.load_artifact(EMBEDDINGS_DIR, embeddings.artifact_name("corpus", name),
                                      allow_non_production=NON_PRODUCTION).select(sample_ids)
    fresh = embeddings.embed(sample_texts, emb, device=DEVICE)
    drift = float(np.max(np.abs(stored - fresh)))
    assert drift <= result["tolerance"], f"{name}: stored artifact differs from a fresh embedding ({drift})"
    print(f"{name}: parity max|diff| {result['max_abs_diff']:.3g} <= {result['tolerance']:.3g} "
          f"| stored vs fresh {drift:.3g}")
pd.DataFrame(report)[["artifact", "status", "shape", "seconds"]]